## Use ridge resgression and cross-validation to train the unmixing model

In [ ]:
# Load packages
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn import linear_model
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pandas as pd
import fcsparser
np.random.seed(233)
import matplotlib.pyplot as plt



In [ ]:
# read data
path = "./path/to/full-stained_raw.fcs"

In [4]:
# parse both metadata and data, get the expression values into a data frame
meta,data = fcsparser.parse(path, meta_data_only=False, reformat_meta=True)

In [ ]:
# have a look
print(type(data))
data

In [ ]:
# read in the spectral matrix
M = pd.read_csv("./spectral_matrix.csv")
print(type(M))

In [ ]:
print(M.shape)
print(data.shape)

In [8]:
# take only the fluorescence channels from the raw data
Y = data.filter(regex="(UV|V|B|YG|R)(\d+)")

# store the other half of the data
C = data.drop(columns=Y.columns)

In [ ]:
print(Y.columns)
print(Y.shape)
print(C.columns)
print(C.shape)

In [10]:
# transpose to have channels as row names
YT = Y.T

In [ ]:
# final checks
print(YT.shape)
print(M.shape)
print(type(M))
print(type(YT))
print(M.columns)

In [16]:
sorted(sklearn.metrics.SCORERS.keys())

['accuracy',
 'adjusted_mutual_info_score',
 'adjusted_rand_score',
 'average_precision',
 'balanced_accuracy',
 'completeness_score',
 'explained_variance',
 'f1',
 'f1_macro',
 'f1_micro',
 'f1_samples',
 'f1_weighted',
 'fowlkes_mallows_score',
 'homogeneity_score',
 'jaccard',
 'jaccard_macro',
 'jaccard_micro',
 'jaccard_samples',
 'jaccard_weighted',
 'max_error',
 'mutual_info_score',
 'neg_brier_score',
 'neg_log_loss',
 'neg_mean_absolute_error',
 'neg_mean_absolute_percentage_error',
 'neg_mean_gamma_deviance',
 'neg_mean_poisson_deviance',
 'neg_mean_squared_error',
 'neg_mean_squared_log_error',
 'neg_median_absolute_error',
 'neg_root_mean_squared_error',
 'normalized_mutual_info_score',
 'precision',
 'precision_macro',
 'precision_micro',
 'precision_samples',
 'precision_weighted',
 'r2',
 'rand_score',
 'recall',
 'recall_macro',
 'recall_micro',
 'recall_samples',
 'recall_weighted',
 'roc_auc',
 'roc_auc_ovo',
 'roc_auc_ovo_weighted',
 'roc_auc_ovr',
 'roc_auc_ovr_we

In [ ]:
# Randomly selecte 75% of the channels to train the model to avoid overfitting. It perfoms better than using all data to train it from my experience.
x_train, x_test, y_train, y_test = train_test_split(M, YT, test_size=0.25, random_state=42)


In [ ]:
print(y_train.shape)
print(x_train.shape)
print(x_test.shape)
print(y_test.shape)

In [ ]:
# pick alpha using CV and build the best model based on smallest MSE

# Candidates generated by smallest positive one from the lg of the d from SVD of my spectra matrix. Then added some smaller values and larger ones on a log scale.
m1 = linear_model.RidgeCV(alphas=[0.1, 0.01, 0.001], scoring = "neg_mean_squared_error",store_cv_results=True) # use LOO CV by default, with negative MSE by default, use poisson deviance if you have strictly all positive values. Uses SVD solver by default when n_features < n_channels

m1.fit(x_train, y_train)
# take the best model's coefficients
print(m1.alpha_)
best_a = m1.coef_

In [ ]:
# Just a quick check
print(m1.score(x_train, y_train))
print(m1.score(x_test, y_test)) 
print(best_a.shape)

In [ ]:

# generate a df
feature_names = M.columns

best_A = pd.DataFrame(
    m1.coef_,
    columns=feature_names
)

unmix = pd.concat([best_A, C], axis=1)

# Write to .csv
unmix.to_csv("name_unmixed.csv", index=False)

## Visualise the alpha choices if you want

In [ ]:
print(m1.cv_results_.shape)
cv_results = m1.cv_results_ # get the all MSEs
# Aggregate MSE per channel by extracting median
med_mse_per_chl = np.median(cv_results, axis=(1))
print(med_mse_per_chl.shape)

# Plotting
plt.figure(figsize=(8, 5))
alphas = [0.1, 0.01, 0.001]
best_alpha = m1.alpha_
for i, mse in enumerate(med_mse_per_chl):
    plt.semilogx(alphas, mse, alpha=0.5, color="blue", linewidth=0.8)
# add elements
plt.axvline(best_alpha, color="red", linestyle="--", label=f"Best alpha = {best_alpha:.3f}")
plt.xlabel("alpha")
plt.ylabel("Channel median MSE")
plt.title("Median MSE vs alphas across training channels")
plt.legend()
plt.show()